In [3]:

!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 joblib==1.5.3 scipy==1.15.3

import pandas as pd
import numpy as np
import re
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


df = pd.read_csv('/content/geo-reviews-dataset-2023.csv',
                 encoding='utf-8',
                 encoding_errors='ignore',
                 on_bad_lines='skip')

print("размер:", df.shape)

if df.iloc[0][0] == 'address':
    df = df.iloc[1:].reset_index(drop=True)

# обработка рейтинга
df['rating'] = df['rating'].astype(str).str.replace('.', '').str.strip()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['rating'])
df['rating'] = (df['rating'].astype(int) / 10).astype(int)


# обработка текста (нижний регистр, символы, пробелы)
def clean_text(text):

    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^а-яё\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.len() > 10]

# 0 - негатив, 1 - нейтрально, 2 - позитив
df['sentiment'] = df['rating'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

print(df['sentiment'].value_counts())

# балансировка классов
min_count = df['sentiment'].value_counts().min()
balanced_dfs = []

for class_num in [0, 1, 2]:
    class_data = df[df['sentiment'] == class_num]
    balanced_dfs.append(class_data.sample(n=min_count, random_state=42))

df_balanced = pd.concat(balanced_dfs)
df_balanced = df_balanced.sample(frac=1, random_state=42)

# подготовка к обуччению
X_train, X_test, y_train, y_test = train_test_split(
    df_balanced['text'],
    df_balanced['sentiment'],
    test_size=0.2,
    random_state=42
)

# векторизация
vectorizer = TfidfVectorizer(max_features=3000, min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# обучение модели
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

accuracy = (model.predict(X_test_vec) == y_test).mean()
print("точность:", accuracy)

# выгрузка и скачивание модели
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))
print('сохранено')

from google.colab import files
files.download('model.pkl')
files.download('vectorizer.pkl')


/tmp/ipykernel_50035/1483886546.py:12: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/geo-reviews-dataset-2023.csv',
/tmp/ipykernel_50035/1483886546.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if df.iloc[0][0] == 'address':


размер: (500074, 5)
sentiment
2    318151
0    165646
1     16095
Name: count, dtype: int64
точность: 0.5708812260536399
сохранено


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>